# B1 — Unconstrained PPO

**Purpose:** Train unconstrained PPO with CjMmCriterion on training days (Jan 1–6);
evaluate on the held-out test set (Jan 7–29).

**Design:**
- Training environment: 6 days concatenated into a single long episode (~4.3M steps).
- Reward: `CjMmCriterion(φ=0.01)` — PnL minus inventory-aversion term.
- Observations normalised via `VecNormalize` (running stats computed during training).
- Test evaluation: per-day, frozen VecNormalize stats from training.

**Outputs:**
- `models/ppo_b1_doge.zip` — trained PPO weights
- `models/vecnorm_b1.pkl` — VecNormalize running statistics
- `results/b1_test_results.csv` — per-day test metrics

In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = next(
    (p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
     if (p / "procs").exists()),
    pathlib.Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from stable_baselines3 import PPO

from procs.gym.experiment_config import ReplayExperimentConfig
from procs.gym.data_loader import load_multi_day
from procs.gym.sb3_wrapper import StableBaselinesTradingEnvironment
from procs.gym.notebook_support import (
    build_multi_day_replay_env,
    make_vecnorm,
    evaluate_rl_per_day,
)
from procs.rewards import CjMmCriterion

cfg = ReplayExperimentConfig()
cfg.ensure_artifact_dirs()
print(f"Repo root : {cfg.repo_root}")
print(f"Models    : {cfg.models_dir}")
print(f"Results   : {cfg.results_dir}")

Repo root : C:\Users\john-\Documents\Thesis_AI4T
Models    : C:\Users\john-\Documents\Thesis_AI4T\models
Results   : C:\Users\john-\Documents\Thesis_AI4T\results


## Section 1 — Data Loading and Train/Test Split

In [2]:
TRAIN_DAYS = 6

daily_S, daily_dt, dates = load_multi_day(str(cfg.datasets_dir), pair=cfg.pair)

train_S     = daily_S[:TRAIN_DAYS]
train_dt    = daily_dt[:TRAIN_DAYS]
train_dates = dates[:TRAIN_DAYS]

test_S      = daily_S[TRAIN_DAYS:]
test_dt     = daily_dt[TRAIN_DAYS:]
test_dates  = dates[TRAIN_DAYS:]


EXPECTED_TEST_DAYS = 23
if len(train_S) != TRAIN_DAYS or len(test_S) != EXPECTED_TEST_DAYS:
    raise ValueError(
        f"Expected {TRAIN_DAYS} train days and {EXPECTED_TEST_DAYS} test days, "
        f"found {len(train_S)} and {len(test_S)}."
    )
if set(map(str, train_dates)) & set(map(str, test_dates)):
    raise ValueError("Train/test date overlap detected.")

# Keep days separate so each environment reset starts a real trading day.
train_snapshots = sum(len(S) for S in train_S)

print(f"Training days : {len(train_S)}  ({train_dates[0]} → {train_dates[-1]})")
print(f"Test days     : {len(test_S)}  ({test_dates[0]} → {test_dates[-1]})")
print(f"Training snapshots: {train_snapshots:,} (~{train_snapshots/1e6:.2f}M)")

  2025-01-01:  713,815 snapshots, σ=0.000021
  2025-01-02:  766,464 snapshots, σ=0.000035
  2025-01-03:  776,383 snapshots, σ=0.000047
  2025-01-04:  778,293 snapshots, σ=0.000045
  2025-01-05:  723,494 snapshots, σ=0.000031
  2025-01-06:  766,311 snapshots, σ=0.000035
  2025-01-07:  787,093 snapshots, σ=0.000062
  2025-01-08:  821,589 snapshots, σ=0.000052
  2025-01-09:  809,421 snapshots, σ=0.000046
  2025-01-10:  789,320 snapshots, σ=0.000045
  2025-01-11:  724,826 snapshots, σ=0.000023
  2025-01-12:  719,550 snapshots, σ=0.000022
  2025-01-13:  819,981 snapshots, σ=0.000046
  2025-01-14:  775,454 snapshots, σ=0.000038
  2025-01-15:  782,299 snapshots, σ=0.000047
  2025-01-16:  788,946 snapshots, σ=0.000048
  2025-01-17:  801,259 snapshots, σ=0.000061
  2025-01-18:  829,125 snapshots, σ=0.000074
  2025-01-19:  842,892 snapshots, σ=0.000096
  2025-01-20:  857,508 snapshots, σ=0.000115
  2025-01-21:  848,107 snapshots, σ=0.000112
  2025-01-22:  812,180 snapshots, σ=0.000044
  2025-01-

## Section 2 — Build Multi-Day Training Environment

In [3]:
reward_fn_train = CjMmCriterion(per_step_inventory_aversion=cfg.phi)

train_env  = build_multi_day_replay_env(train_S, train_dt, cfg, reward_fn=reward_fn_train, mode="sequential")
train_sb3  = StableBaselinesTradingEnvironment(train_env)
train_vn   = make_vecnorm(train_sb3, cfg, training=True, norm_reward=True)

obs_dim = train_env.observation_space.shape[0]
print(f"Observation dim : {obs_dim}")
print(f"Action dim      : {train_env.action_space.shape[0]}")
print(f"Training days   : {len(train_S)} separate replay episodes")
print(f"Snapshots total : {train_snapshots:,}")

Observation dim : 5
Action dim      : 2
Training days   : 6 separate replay episodes
Snapshots total : 4,524,760


## Section 3 — PPO Training

**`TOTAL_TIMESTEPS` guide:**
- Quick local test: `200_000`
- Full Snellius run: `max(2 * len(S_train), 2_000_000)`

In [4]:
TOTAL_TIMESTEPS = max(train_snapshots, 1_000_000)  # ≥1 full pass or 1M steps
print(f"TOTAL_TIMESTEPS = {TOTAL_TIMESTEPS:,}  ({TOTAL_TIMESTEPS/train_snapshots:.2f}× data)")

model_b1 = PPO(
    "MlpPolicy",
    train_vn,
    **cfg.ppo_kwargs(),
    tensorboard_log=str(cfg.repo_root / "tb_logs" / "b1"),
    verbose=1,
    device="cpu",
)

print(f"PPO kwargs: {cfg.ppo_kwargs()}")
model_b1.learn(total_timesteps=TOTAL_TIMESTEPS)

TOTAL_TIMESTEPS = 4,524,760  (1.00× data)
Using cpu device
PPO kwargs: {'n_steps': 2048, 'batch_size': 512, 'n_epochs': 10, 'learning_rate': 0.0003, 'gamma': 0.999, 'gae_lambda': 0.95, 'clip_range': 0.2, 'ent_coef': 0.01}
Logging to C:\Users\john-\Documents\Thesis_AI4T\tb_logs\b1\PPO_1
-----------------------------
| time/              |      |
|    fps             | 1175 |
|    iterations      | 1    |
|    time_elapsed    | 1    |
|    total_timesteps | 2048 |
-----------------------------
-----------------------------------------
| time/                   |             |
|    fps                  | 945         |
|    iterations           | 2           |
|    time_elapsed         | 4           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.006261553 |
|    clip_fraction        | 0.0203      |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.84       |
|    explained_variance   | -0.155      |

In [5]:
model_b1.save(str(cfg.model_path("ppo_b1_doge")))
train_vn.save(str(cfg.vecnorm_path("vecnorm_b1")))
print(f"Saved model  → {cfg.model_path('ppo_b1_doge')}.zip")
print(f"Saved vecnorm → {cfg.vecnorm_path('vecnorm_b1')}")

Saved model  → C:\Users\john-\Documents\Thesis_AI4T\models\ppo_b1_doge.zip
Saved vecnorm → C:\Users\john-\Documents\Thesis_AI4T\models\vecnorm_b1.pkl


## Section 4 — Training Diagnostics

Quick diagnostic from the last logged progress (PPO prints every `n_steps` rollout).
For full curves, run: `tensorboard --logdir tb_logs/b1`

In [6]:
# Parse TensorBoard event file for explained variance and entropy
try:
    from tensorflow.python.summary.summary_iterator import summary_iterator
    tb_dir = cfg.repo_root / "tb_logs" / "b1"
    ev_files = sorted(tb_dir.glob("events.out.tfevents.*"))
    if ev_files:
        ev_data = {"step": [], "explained_variance": [], "entropy_loss": []}
        for event in summary_iterator(str(ev_files[-1])):
            for v in event.summary.value:
                if v.tag == "train/explained_variance":
                    ev_data["step"].append(event.step)
                    ev_data["explained_variance"].append(v.simple_value)
                elif v.tag == "train/entropy_loss":
                    ev_data["entropy_loss"].append(v.simple_value)

        df_tb = pd.DataFrame(ev_data).dropna()
        if len(df_tb):
            fig, axes = plt.subplots(1, 2, figsize=(12, 4))
            axes[0].plot(df_tb["step"], df_tb["explained_variance"])
            axes[0].axhline(0.5, color="red", linestyle="--", alpha=0.5, label="target ≥ 0.5")
            axes[0].set_title("Explained Variance")
            axes[0].legend(); axes[0].grid(True, alpha=0.3)
            axes[1].plot(df_tb["step"], df_tb.get("entropy_loss", []))
            axes[1].set_title("Entropy Loss")
            axes[1].grid(True, alpha=0.3)
            plt.suptitle("B1 Training Diagnostics")
            plt.tight_layout()
            plt.show()
        print(f"Final explained variance: {df_tb['explained_variance'].iloc[-1]:.4f}")
    else:
        print("No TensorBoard event files found — skipping diagnostic plot.")
except Exception as e:
    print(f"TensorBoard parsing unavailable ({e}). Run: tensorboard --logdir tb_logs/b1")

TensorBoard parsing unavailable (No module named 'tensorflow'). Run: tensorboard --logdir tb_logs/b1


## Section 5 — Test Evaluation (Days 7–29)

In [9]:
model_b1_loaded = PPO.load(str(cfg.model_path("ppo_b1_doge")), device="cpu")

df_b1 = evaluate_rl_per_day(
    model=model_b1_loaded,
    vecnorm_path=cfg.vecnorm_path("vecnorm_b1"),
    test_S=test_S,
    test_dt=test_dt,
    test_dates=test_dates,
    config=cfg,
    seed=42,
    num_rollouts=1,
)

print(f"Evaluated {len(df_b1)} test days.")

Evaluated 23 test days.


## Section 6 — Results Display and Save

In [10]:
pd.set_option("display.float_format", "{:.4f}".format)
METRICS = ["Sharpe", "Sortino", "Max DD", "P&L-to-MAP", "Final PnL"]

summary = pd.DataFrame({
    "Mean":   df_b1[METRICS].mean(),
    "Std":    df_b1[METRICS].std(),
    "Median": df_b1[METRICS].median(),
}).T

print("=== B1 Per-day Test Results ===")
print(df_b1[METRICS].to_string())
print("\n=== B1 Summary ===")
print(summary.to_string())

=== B1 Per-day Test Results ===
            Sharpe  Sortino  Max DD  P&L-to-MAP  Final PnL
Day                                                       
2025-01-07  0.0140   0.0040  0.0133      0.7688     0.1418
2025-01-08  0.0094   0.0045  0.0105      0.2058     0.0973
2025-01-09  0.0091   0.0049  0.0096      0.1646     0.0838
2025-01-10  0.0087   0.0036  0.0062      0.1741     0.0847
2025-01-11  0.0200   0.0082  0.0024      0.2199     0.0977
2025-01-12  0.0194   0.0076  0.0031      0.2014     0.0944
2025-01-13  0.0114   0.0056  0.0057      0.2269     0.1071
2025-01-14  0.0135   0.0053  0.0053      0.2418     0.0974
2025-01-15  0.0133   0.0047  0.0091      0.3286     0.1111
2025-01-16  0.0183   0.0059  0.0082      0.5395     0.1397
2025-01-17  0.0148   0.0038  0.0119      0.5974     0.1162
2025-01-18  0.0152   0.0046  0.0076      1.4100     0.1558
2025-01-19  0.0078   0.0033  0.0107      0.3964     0.1198
2025-01-20  0.0087   0.0038  0.0115      0.7195     0.2021
2025-01-21  0.0045   0.0

In [11]:
out_path = cfg.result_path("b1_test_results.csv")
df_b1.to_csv(out_path)
print(f"Saved → {out_path}")

Saved → C:\Users\john-\Documents\Thesis_AI4T\results\b1_test_results.csv
